# Notebook 12 — Organization-Level Investigation Queue and Explainability

**Purpose:** Turn Notebook 11's filing-level research rankings into an auditable, one-record-per-EIN 2024 review queue for a criminal investigations team. Provide financial context, peer-relative comparisons, historical trends, and separate model signals. **This is a screening/research tool, not a finding or probability of fraud.**

**Inputs:** Notebook 11 primary and limited-data Parquet queues; revised Notebook 10 anomaly queue; 2010–2024 longitudinal feature Parquet. **Outputs:** `data/processed/notebook12/`. No existing files are overwritten outside this folder.

**Design:** Keep the 50/50 Notebook 11 consensus as an *exploratory sorting reference*, not a validated fraud-risk score. Do not use the disclosure label to determine ranks. Do not infer wrongdoing from financial outliers. Review with investigators before operational use.

## 1. Configuration and input checks

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import duckdb
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'
IN11_PRIMARY = DATA / 'notebook11' / '06_2024_integrated_primary_queue.parquet'
IN11_LIMITED = DATA / 'notebook11' / '06_2024_limited_data_queue.parquet'
IN10 = DATA / 'notebook10' / '06_2024_form990_anomaly_queue.parquet'
FEATURES = DATA / 'nccs_longitudinal_features_v1.parquet'
OUT = DATA / 'notebook12'
OUT.mkdir(parents=True, exist_ok=True)
TOP_K = (100, 500, 1000)
for path in (IN11_PRIMARY, IN11_LIMITED, IN10, FEATURES):
    assert path.is_file(), f'Missing prerequisite: {path}'

def sqlpath(path):
    return str(path).replace("'", "''")

def finite_numeric(s):
    return pd.to_numeric(s, errors='coerce').replace([np.inf, -np.inf], np.nan)

con = duckdb.connect()
con.execute('SET threads=4')
con.execute("SET memory_limit='6GB'")
for name, path in [('p11', IN11_PRIMARY), ('l11', IN11_LIMITED), ('a10', IN10), ('features', FEATURES)]:
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{sqlpath(path)}')")
print('Project:', ROOT)
print('Output:', OUT)

## 2. Verify source integrity and identify multiple filings per EIN

Notebook 11 ranks filing IDs (`OBJECTID`), whereas investigators need **one record per organization (`ein`)**. Audit duplicates before selection. Choose the highest-priority eligible 2024 filing per EIN (primary over limited; then Notebook 11 consensus rank; then OBJECTID as a deterministic tie-breaker). This is a *display-selection rule*, not a claim that the selected filing is the latest amendment. Retain all filing counts in the audit.

In [ ]:
primary = con.execute('SELECT * FROM p11').df()
limited = con.execute('SELECT * FROM l11').df()
assert set(primary.columns) == set(limited.columns), 'Notebook 11 queues have incompatible schemas'
q = pd.concat([primary, limited], ignore_index=True)
assert q.OBJECTID.notna().all() and q.OBJECTID.is_unique
assert q.tax_year.eq(2024).all()
assert q.integration_cohort.isin(['primary_review','limited_data']).all()
q['ein'] = (
    q['ein']
    .astype('string')
    .str.strip()
    .str.replace(r'^EIN-', '', regex=True)
    .str.replace('-', '', regex=False)
    .str.replace(r'\.0$', '', regex=True)
    .str.zfill(9)
)
assert q.ein.str.fullmatch(r'\d{9}').all(), 'Unexpected EIN format; investigate before deduplicating'
q['consensus_rank'] = pd.to_numeric(
    q['consensus_rank'], errors='coerce'
).astype('float64')

q['selection_priority'] = np.where(
    q.integration_cohort.eq('primary_review'), 0, 1
)

q['selection_rank'] = q['consensus_rank'].fillna(np.inf)
q['objectid_sort'] = q.OBJECTID.astype(str)
q = q.sort_values(['ein','selection_priority','selection_rank','objectid_sort'],kind='mergesort')
counts = q.groupby('ein',dropna=False).size().rename('filings_for_ein')
selected = q.drop_duplicates('ein',keep='first').copy().merge(counts, on='ein', validate='one_to_one')
assert selected.ein.is_unique
assert len(selected) == q.ein.nunique()
dup_summary = pd.DataFrame([{'filing_rows':len(q),'unique_eins':len(selected),
    'eins_with_multiple_filings':int((counts>1).sum()),
    'extra_filing_rows':int((counts-1).clip(lower=0).sum()),
    'selected_primary':int(selected.integration_cohort.eq('primary_review').sum()),
    'selected_limited':int(selected.integration_cohort.eq('limited_data').sum())}])
display(dup_summary)
dup_summary.to_csv(OUT/'01_ein_dedup_summary.csv',index=False)
q.loc[q.ein.isin(counts[counts>1].index), ['ein','OBJECTID','integration_cohort','consensus_rank','filings_for_ein'] if 'filings_for_ein' in q.columns else ['ein','OBJECTID','integration_cohort','consensus_rank']].to_csv(OUT/'01_duplicate_ein_filing_audit.csv',index=False)

## 3. Add financial and governance context from Notebook 10

Join on **OBJECTID**, not EIN, to avoid mixing values from different filings. Keep source metrics separate from risk-indicator severities. Peer groups are revenue-based and do not account for nonprofit mission or subsector.

In [ ]:
a_cols = set(con.execute('DESCRIBE a10').df().column_name)
context_candidates = ['expense_revenue_ratio','liabilities_assets_ratio','net_margin',
    'revenue_growth_3yr','asset_growth_3yr','revenue_volatility_3yr','board_independence',
    'peer_group','isolation_anomaly_score']
context_cols = [c for c in context_candidates if c in a_cols and c not in selected.columns]
ctx = con.execute('SELECT OBJECTID' + ''.join(f', "{c}"' for c in context_cols) + ' FROM a10').df()
assert ctx.OBJECTID.is_unique
selected = selected.merge(ctx, on='OBJECTID',how='left',validate='one_to_one',indicator='_context_match')
assert selected._context_match.eq('both').all(), 'Notebook 10 context did not match all selected filings'
selected.drop(columns='_context_match',inplace=True)
print('Context fields added:',context_cols)

## 4. Historical organization trends (2022–2024)

Use the deduplicated longitudinal features. For each EIN/year, take the row with the selected 2024 OBJECTID when applicable, otherwise use deterministic OBJECTID sorting for historical display. Historical comparisons are descriptive and **not guaranteed to have been known at a past scoring date**.

In [ ]:
fcols = set(con.execute('DESCRIBE features').df().column_name)

base_needed = {
    'OBJECTID', 'ein', 'tax_year',
    'total_revenue', 'total_expenses',
    'total_assets', 'total_liabilities', 'net_income'
}

assert base_needed.issubset(fcols), sorted(base_needed - fcols)

# The investigation queue uses nine-digit EINs.
# Normalize historical EINs to the same format before joining.
con.register('selected_eins', selected[['ein']].copy())

hist = con.execute("""
WITH normalized_history AS (
    SELECT
        LPAD(
            REGEXP_REPLACE(
                REGEXP_REPLACE(
                    TRIM(CAST(ein AS VARCHAR)),
                    '^EIN-',
                    '',
                    'i'
                ),
                '[^0-9]',
                '',
                'g'
            ),
            9,
            '0'
        ) AS ein,
        tax_year,
        OBJECTID,
        TRY_CAST(total_revenue AS DOUBLE) AS revenue,
        TRY_CAST(total_expenses AS DOUBLE) AS expenses,
        TRY_CAST(total_assets AS DOUBLE) AS assets,
        TRY_CAST(total_liabilities AS DOUBLE) AS liabilities,
        TRY_CAST(net_income AS DOUBLE) AS net_income
    FROM features
    WHERE tax_year BETWEEN 2022 AND 2024
)
SELECT h.*
FROM normalized_history h
INNER JOIN selected_eins s
    ON h.ein = s.ein
WHERE REGEXP_FULL_MATCH(h.ein, '[0-9]{9}')
""").df()

# Confirm the join returned actual historical records.
assert len(hist) > 0, (
    'Historical join returned zero rows. Check EIN normalization.'
)

hist['ein'] = hist['ein'].astype('string')
hist['OBJECTID'] = hist['OBJECTID'].astype(str)

# Keep one historical filing per EIN and tax year.
hist = (
    hist.sort_values(
        ['ein', 'tax_year', 'OBJECTID'],
        kind='mergesort'
    )
    .drop_duplicates(['ein', 'tax_year'], keep='last')
)

assert not hist.duplicated(['ein', 'tax_year']).any()

# Convert historical data into one row per organization.
wide = hist.pivot(
    index='ein',
    columns='tax_year',
    values=['revenue', 'expenses', 'assets', 'liabilities', 'net_income']
)

wide.columns = [
    f'{metric}_{year}'
    for metric, year in wide.columns
]

wide = wide.reset_index()

selected = selected.merge(
    wide,
    on='ein',
    how='left',
    validate='one_to_one'
)

# Calculate 2022–2024 changes.
for metric in ['revenue', 'assets', 'net_income']:
    a = f'{metric}_2022'
    b = f'{metric}_2024'

    if a in selected.columns and b in selected.columns:
        denom = selected[a].abs().where(
            selected[a].abs() > 1e-9
        )

        selected[f'{metric}_change_2022_2024_pct'] = (
            100 * (selected[b] - selected[a]) / denom
        ).replace([np.inf, -np.inf], np.nan)

# Validate historical coverage.
coverage = pd.DataFrame([
    {
        'tax_year': year,
        'organizations_with_history': int(
            selected[f'revenue_{year}'].notna().sum()
        ),
        'coverage_pct': round(
            100 * selected[f'revenue_{year}'].notna().mean(),
            2
        )
    }
    for year in [2022, 2023, 2024]
])

display(coverage)

assert selected['revenue_2024'].notna().sum() > 0, (
    'Historical 2024 revenue is still missing.'
)

print('Historical context joined successfully.')
print('Organizations:', len(selected))
print('Historical records matched:', len(hist))

## 5. Peer-relative benchmarks and explanatory indicators

Calculate empirical percentile ranks within Notebook 10 revenue peer groups for selected metrics. **Percentiles describe relative position, not fraud likelihood.** Keep numerator/denominator context visible; very small revenue can make expense ratios unstable. Do not assert that any indicator is itself evidence of wrongdoing.

In [ ]:
peer_metrics = [c for c in ['expense_revenue_ratio','liabilities_assets_ratio',
    'revenue_growth_3yr','asset_growth_3yr','revenue_volatility_3yr','board_independence'] if c in selected.columns]
for c in peer_metrics:
    selected[c]=finite_numeric(selected[c])
    selected[f'{c}_peer_percentile']=selected.groupby('peer_group',dropna=False)[c].rank(pct=True,method='average')

labels = {
 'expense_revenue_ratio':'Expenses high relative to revenue',
 'liabilities_assets_ratio':'Liabilities high relative to assets',
 'revenue_growth_3yr':'Three-year revenue growth unusually low',
 'asset_growth_3yr':'Three-year asset growth unusually low',
 'revenue_volatility_3yr':'Revenue volatility unusually high',
 'board_independence':'Board independence unusually low'
}
SIX = list(labels)
contribution_cols = ['adverse_'+x for x in SIX]
assert set(contribution_cols).issubset(selected.columns)
for c in contribution_cols:
    selected[c]=finite_numeric(selected[c])

def explain(row, max_items=3):
    vals=[(float(row['adverse_'+c]),labels[c]) for c in SIX if pd.notna(row['adverse_'+c])]
    vals=sorted((v,label) for v,label in vals if v>0)
    vals.reverse()
    return '; '.join(label for _,label in vals[:max_items]) if vals else 'No positive adverse peer indicator available'
selected['top_peer_explanations']=selected.apply(explain,axis=1)
selected['observed_adverse_indicator_count']=selected[contribution_cols].gt(0).sum(axis=1)
print('Peer metrics:',peer_metrics)

## 6. Organization-level rankings and agreement flags

Re-rank the *selected organizations* rather than reusing filing ranks as if they were organization ranks. Preserve original Notebook 11 filing ranks separately for traceability. A cross-method flag means two **statistical** screens prioritize the same organization, not that two independent sources corroborate misconduct; the models share input data.

In [ ]:
selected['filing_consensus_rank_11']=selected['consensus_rank']
selected['filing_supervised_rank_11']=selected['supervised_rank']
selected['filing_peer_rank_11']=selected['peer_rank']
selected['filing_isolation_rank_11']=selected['isolation_rank']
org_primary=selected.integration_cohort.eq('primary_review')
for name, col in [('consensus','consensus_score'),('supervised','supervised_diversion_score'),
                  ('peer','peer_adverse_6'),('isolation','isolation_anomaly_score')]:
    selected[f'organization_{name}_rank']=pd.Series(pd.NA,index=selected.index,dtype='Int64')
    valid=org_primary & finite_numeric(selected[col]).notna()
    # Tie-break consistently by EIN, avoiding dependence on arbitrary row order.
    ranked=selected.loc[valid,['ein',col]].sort_values([col,'ein'],ascending=[False,True],kind='mergesort')
    selected.loc[ranked.index,f'organization_{name}_rank']=np.arange(1,len(ranked)+1)

for k in TOP_K:
    selected[f'top{k}_supervised']=selected.organization_supervised_rank.le(k).fillna(False)
    selected[f'top{k}_peer']=selected.organization_peer_rank.le(k).fillna(False)
    selected[f'top{k}_consensus']=selected.organization_consensus_rank.le(k).fillna(False)
    selected[f'top{k}_both_methods']=selected[f'top{k}_supervised'] & selected[f'top{k}_peer']
selected['signal_pattern_top1000']=np.select(
    [selected.top1000_both_methods,selected.top1000_supervised,selected.top1000_peer],
    ['Both supervised and peer top 1000','Supervised top 1000 only','Peer top 1000 only'],
    default='Neither component top 1000')
selected.loc[~org_primary,'signal_pattern_top1000']='Limited data - no comparable primary ranking'
selected['screening_notice']='Research prioritization only; not an allegation, proof, or calibrated probability of fraud.'
selected['label_notice']='Supervised model was trained on self-reported diversion disclosures, not confirmed fraud.'
selected['data_notice']=np.where(org_primary,
    'Meets six-indicator coverage policy; peer groups based on revenue only.',
    'Insufficient peer-indicator coverage for primary ranking; review separately.')
summary=selected.groupby(['integration_cohort','signal_pattern_top1000'],dropna=False).size().rename('organizations').reset_index()
display(summary)
summary.to_csv(OUT/'06_signal_agreement_summary.csv',index=False)

## 7. Export analyst-friendly research queues

The primary queue is ordered by the **exploratory** Notebook 11 consensus score at the organization level; it retains independent component ranks and explanations. The limited-data queue is not assigned a comparable consensus rank. Export full Parquet datasets and manageable CSV previews. The disclosure outcome is deliberately excluded from investigator-facing exports to avoid suggesting it is verified fraud or using it as a lead justification.

In [ ]:
export_cols = [
 'ein','OBJECTID','tax_year','filings_for_ein','peer_group','integration_cohort',
 'six_indicator_count','available_indicator_count','revenue','expenses','assets','liabilities','net_income',
 'supervised_diversion_score','peer_adverse_6','isolation_anomaly_score','consensus_score',
 'organization_consensus_rank','organization_supervised_rank','organization_peer_rank','organization_isolation_rank',
 'filing_consensus_rank_11','filing_supervised_rank_11','filing_peer_rank_11',
 'signal_pattern_top1000','top1000_both_methods','top_peer_explanations',
 'observed_adverse_indicator_count']
export_cols += peer_metrics + [f'{c}_peer_percentile' for c in peer_metrics]
export_cols += contribution_cols
export_cols += [c for c in selected.columns if c.endswith(('_2022','_2023','_2024','_2022_2024_pct'))]
export_cols += ['screening_notice','label_notice','data_notice']
export_cols=list(dict.fromkeys(c for c in export_cols if c in selected.columns))
assert 'fraud_label' not in export_cols
primary_out=selected.loc[org_primary,export_cols].sort_values('organization_consensus_rank').copy()
limited_out=selected.loc[~org_primary,export_cols].sort_values(['six_indicator_count','ein'],ascending=[False,True]).copy()
primary_out.to_parquet(OUT/'07_2024_organization_primary_queue.parquet',index=False)
limited_out.to_parquet(OUT/'07_2024_organization_limited_data_queue.parquet',index=False)
primary_out.head(1000).to_csv(OUT/'07_top1000_organization_research_leads.csv',index=False)
limited_out.head(1000).to_csv(OUT/'07_limited_data_preview.csv',index=False)
display(primary_out[['organization_consensus_rank','ein','peer_group','six_indicator_count',
 'organization_supervised_rank','organization_peer_rank','top_peer_explanations']].head(20))
print('Primary organizations:',len(primary_out),'Limited-data organizations:',len(limited_out))

## 8. Quality assurance and ranking diagnostics

Verify that no organization appears twice, every input filing is represented by an EIN, primary ranks are continuous, and limited-data organizations are not assigned primary ranks. Review cross-method overlap without treating agreement as validation.

In [ ]:
assert selected.ein.is_unique
assert len(selected)==q.ein.nunique()
assert len(primary_out)+len(limited_out)==len(selected)
assert primary_out.organization_consensus_rank.notna().all()
assert limited_out.organization_consensus_rank.isna().all()
assert primary_out.six_indicator_count.ge(4).all()
assert sorted(primary_out.organization_consensus_rank.astype(int))==list(range(1,len(primary_out)+1))
assert selected.loc[org_primary,'consensus_score'].notna().all()
assert selected.OBJECTID.is_unique
assert set(selected.OBJECTID).issubset(set(q.OBJECTID))
qa={'input_filing_rows':int(len(q)), 'output_unique_eins':int(len(selected)),
    'extra_filing_rows_collapsed':int(len(q)-len(selected)),
    'primary_organizations':int(len(primary_out)), 'limited_data_organizations':int(len(limited_out)),
    'primary_missing_consensus':int(primary_out.consensus_score.isna().sum()),
    'limited_with_consensus_rank':int(limited_out.organization_consensus_rank.notna().sum()),
    'duplicate_output_eins':int(selected.ein.duplicated().sum())}
display(pd.DataFrame([qa]))
(OUT/'08_quality_checks.json').write_text(json.dumps(qa,indent=2))
rows=[]
for k in TOP_K:
    sup=set(selected.loc[selected[f'top{k}_supervised'],'ein'])
    peer=set(selected.loc[selected[f'top{k}_peer'],'ein'])
    combined=set(selected.loc[selected[f'top{k}_consensus'],'ein'])
    rows.append({'top_k':k,'supervised_peer_overlap':len(sup&peer),
                 'supervised_consensus_overlap':len(sup&combined),
                 'peer_consensus_overlap':len(peer&combined)})
display(pd.DataFrame(rows))
pd.DataFrame(rows).to_csv(OUT/'08_organization_rank_overlap.csv',index=False)

## 9. Reproducibility manifest and interpretation

Document selection rules, input artifacts, feature limitations, and the distinction between statistical screening and criminal evidence. Notebook 13 should evaluate sensitivity, potential false positives, and investigative usefulness with independent case outcomes where available.

In [ ]:
manifest={
 'notebook':'12 — Organization-Level Investigation Queue and Explainability',
 'tax_year':2024,
 'input_primary':str(IN11_PRIMARY),'input_limited':str(IN11_LIMITED),
 'input_anomaly_context':str(IN10),'input_historical_features':str(FEATURES),
 'filing_selection':'Per EIN, primary before limited, then lowest Notebook 11 consensus rank, then OBJECTID; not an amendment chronology rule',
 'organization_ranks':'Recomputed within selected primary organizations; highest component score first; EIN tie-break',
 'consensus':'Exploratory Notebook 11 50/50 supervised/peer rank-percentile blend, not a fraud probability',
 'supervised_target':'Self-reported significant diversion of assets, not independently confirmed fraud',
 'peer_comparison':'Within revenue-based peer group; no mission/subsector adjustment',
 'historical_context':'2022-2024 descriptive snapshots; no point-in-time availability guarantee',
 'limitations':[
  'Outliers and governance patterns are not evidence of fraud',
  'The supervised model predicts diversion disclosure, not criminal conduct',
  'Models share inputs; agreement is not independent corroboration',
  'Consensus weighting and top-K thresholds have not been validated against investigator outcomes',
  'Filing selection is a display priority rule; verify amendments before operational use',
  'Revenue-only peer groups may be inappropriate across nonprofit subsectors',
  'Limited-data organizations and Form 990-EZ require separate treatment',
  'The 2023-2024 period has informed model development and is not an untouched holdout',
  'No allegation or enforcement action should be based solely on these rankings'
 ]}
(OUT/'12_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Notebook 12 complete. Saved outputs to:',OUT)

## Final Results, Interpretation, and Next Steps

### 1. Overview

Notebook 12 transformed the integrated risk signals developed in Notebooks 09–11 into an organization-level investigation queue designed to support the identification and prioritization of nonprofit organizations exhibiting potentially suspicious financial or governance behavior.

The notebook combined:

- Supervised machine-learning rankings based on historical self-reported significant diversions of assets.
- Peer-based financial and governance anomaly indicators.
- Isolation Forest anomaly-detection results.
- Historical financial trends from 2022–2024.
- Peer-relative financial comparisons.
- Interpretable explanations of adverse financial and governance indicators.

The objective was to produce a structured, explainable research queue that criminal investigators could use to prioritize organizations for further examination.

The investigation queue is not intended to establish that an organization has committed fraud.

### 2. Final Investigation Population

The analysis included **355,973 organizations from tax year 2024**.

| Population | Organizations |
|---|---:|
| Primary investigation queue | 329,440 |
| Limited-data queue | 26,533 |
| **Total** | **355,973** |

Organization-level deduplication confirmed that each filing represented a unique EIN. No duplicate organizations needed to be collapsed.

All organizations were assigned to exactly one queue.

The primary investigation queue includes organizations with sufficient financial and governance indicator coverage to support the exploratory combined ranking.

Organizations with insufficient indicator coverage were retained in a separate limited-data population rather than excluded from the analysis.

This distinction prevents organizations with missing information from being treated as directly comparable to organizations with more complete financial and governance data.

### 3. Agreement Between Risk-Identification Methods

The supervised model and peer-anomaly framework continued to identify substantially different organizations.

| Classification | Organizations |
|---|---:|
| Top 1,000 in both approaches | 32 |
| Top 1,000 in supervised only | 968 |
| Top 1,000 in peer anomalies only | 968 |
| Neither top 1,000 | 327,472 |
| Limited-data population | 26,533 |
| **Total** | **355,973** |

Only **32 organizations** appeared in the top 1,000 of both approaches.

This reinforces the finding from Notebook 11 that supervised and peer-anomaly rankings capture different patterns within the nonprofit population.

The supervised model identifies characteristics associated with historical diversion disclosures, while peer-anomaly detection identifies potentially concerning financial and governance patterns relative to similarly sized organizations.

Organizations identified by both approaches may warrant additional analytical attention because multiple methods highlight their characteristics.

However, the methods share underlying financial inputs. Agreement between them does not constitute independent evidence of fraud.

### 4. Investigation Queue Features and Explainability

The organization-level investigation queue includes the following information.

**Model rankings**

- Supervised model ranking.
- Peer-anomaly ranking.
- Isolation Forest ranking.
- Exploratory combined ranking.

These rankings are retained separately to allow investigators to distinguish between different analytical signals.

**Financial indicators**

- Total revenue.
- Total expenses.
- Total assets.
- Total liabilities.
- Net income.
- Financial ratios and other available anomaly indicators.

**Governance indicators**

- Board independence characteristics.
- Available governance measures.
- Governance-related anomaly indicators.

**Historical financial context**

- Revenue, expenses, assets, liabilities, and net income for 2022–2024, where available.
- Changes in selected financial metrics across the historical period.
- Historical financial information supporting the interpretation of current-year anomalies.

**Peer comparisons**

- Relative financial and governance characteristics compared with similarly sized nonprofits.
- Peer percentiles for available anomaly indicators.

**Model agreement**

- Organizations appearing in the top 1,000 of both primary approaches.
- Organizations appearing only in the supervised top 1,000.
- Organizations appearing only in the peer-anomaly top 1,000.
- Organizations appearing in neither group.

**Explainability**

- Up to three adverse peer indicators contributing to each organization's anomaly assessment.
- Supporting financial and governance context.
- Data-coverage information and interpretive limitations.

These features allow investigators to examine the underlying reasons an organization was prioritized rather than relying exclusively on a single risk score.

### 5. Historical Financial Data Coverage — Corrected and Validated

The initial version of Notebook 12 contained an EIN-formatting mismatch between the investigation queue and historical NCCS source data.

The investigation queue used normalized nine-digit EINs, while historical records retained the `EIN-XX-XXXXXXX` format.

This caused historical financial values to be missing from the initial investigation queue.

The historical join was corrected by standardizing EINs consistently before matching records.

Notebook 13 subsequently validated the corrected outputs against the original NCCS source.

**Final historical coverage**

| Tax year | Revenue available | Assets available |
|---|---:|---:|
| 2022 | 332,460 | 312,243 |
| 2023 | 343,282 | 330,029 |
| 2024 | 355,973 | 355,973 |

Additional validation confirmed:

- **1,031,943 historical filing records** matched the investigation population.
- **330,550 organizations** had source filings in all three years.
- No duplicate EIN/year combinations were detected in the matched historical records.
- No historical export discrepancies remained after the correction.

The corrected investigation queue now includes usable historical financial context where the necessary source records are available.

Historical completeness remains lower for earlier years, particularly for assets.

Missing historical financial values should be interpreted as unavailable data, not as zero revenue, assets, or financial activity.

### 6. Data Quality and Integrity Checks

Notebook 12 and the subsequent Notebook 13 validation confirmed the following:

| Validation check | Result |
|---|---|
| Total unique organizations | 355,973 |
| Duplicate EINs | 0 |
| Primary investigation queue | 329,440 |
| Limited-data queue | 26,533 |
| Missing primary consensus rankings | 0 |
| Historical export discrepancies | 0 |
| Historical EIN/year duplicates | 0 |
| Investigation queue exports | Successful |

The notebook successfully generated the primary investigation queue, limited-data queue, and a top-1,000 research-lead export.

These checks establish that the organization-level output is structurally consistent and that the corrected historical information was transferred successfully.

They do not establish the accuracy of the risk rankings or their effectiveness in identifying criminal activity.

### 7. Key Findings

**Finding 1: Organization-level investigative prioritization is technically feasible.**

The modeling outputs can be consolidated into a single organization-level dataset that preserves financial context, model rankings, and explanatory indicators.

**Finding 2: Supervised and anomaly-based approaches identify substantially different organizations.**

Only 32 organizations appeared in the top 1,000 of both primary approaches.

This suggests that the approaches provide different perspectives on potentially concerning organizational characteristics.

**Finding 3: Explainability improves the interpretability of investigation leads.**

Financial trends, peer comparisons, and adverse indicators provide a basis for understanding why an organization was prioritized.

These explanations can help investigators distinguish potentially meaningful anomalies from ordinary organizational differences.

**Finding 4: Historical financial information provides important context.**

The corrected historical join allows investigators to examine financial changes over multiple years rather than relying exclusively on a single tax-year snapshot.

However, historical availability varies by organization and financial measure.

**Finding 5: Data completeness must remain visible.**

Organizations with insufficient peer-indicator coverage are retained separately to avoid misleading comparisons.

**Finding 6: The supervised ranking may be strongly associated with organization size.**

Notebook 13 found substantial differences in median revenue between organizations prioritized by the supervised model and those prioritized by peer-anomaly detection.

The supervised-only top-1,000 group had median annual revenue of approximately $407 million, compared with approximately $650,000 for the peer-only group.

This raises concerns about whether organization size disproportionately influences the supervised ranking.

The relationship requires further analysis before the rankings are considered for operational use.

### 8. Remaining Limitations

The following limitations remain:

1. **No independently confirmed fraud labels:** The supervised model was trained using self-reported significant diversions of assets rather than confirmed criminal fraud outcomes.

2. **Exploratory ranking weights:** The combined ranking uses weights that have not been validated against investigative results.

3. **Potential organization-size effects:** The supervised model may disproportionately prioritize large nonprofits.

4. **Peer-group limitations:** Peer comparisons are primarily based on revenue size rather than nonprofit mission or subsector.

5. **Incomplete historical coverage:** Some organizations lack prior-year financial records or specific historical measures.

6. **Extreme financial values:** Unusual financial ratios or large values may reflect legitimate organizational characteristics, accounting practices, or reporting errors.

7. **Shared underlying inputs:** Agreement between supervised and anomaly-based methods does not establish independent corroboration.

8. **No completed investigator review:** The current queue has not been evaluated against structured investigator assessments or independently confirmed outcomes.

9. **Unvalidated operational performance:** The rankings have not demonstrated prospective effectiveness in identifying actionable criminal investigations.

**The investigation queue represents analytical research leads, not validated fraud predictions, evidence of misconduct, or allegations against individual organizations.**

### 9. Next Steps

Notebook 13 evaluated the investigation queue's historical coverage, ranking stability, data quality, and potential investigative usefulness.

It confirmed that the corrected historical data pipeline is functioning properly while identifying important methodological concerns.

The next phase is:

**Notebook 14 — Organization Size Bias and Model Robustness**

Notebook 14 should:

1. Examine supervised model scores and rankings across nonprofit revenue bands.
2. Evaluate relationships between organization size, financial characteristics, and supervised model scores.
3. Compare model ranking behavior among organizations of similar sizes.
4. Investigate whether organization-size effects disproportionately influence the highest-ranked organizations.
5. Evaluate size-adjusted ranking approaches as sensitivity analyses.
6. Compare organizations prioritized by supervised methods, peer anomalies, and both approaches.
7. Assess whether alternative ranking approaches produce more stable and interpretable investigative priorities.

These analyses should preserve the existing model outputs for comparison rather than automatically replacing the current rankings.

### 10. Final Conclusion

Notebook 12 successfully transformed the supervised and anomaly-detection results into an explainable, organization-level investigation queue containing **355,973 nonprofits**, including **329,440 organizations** in the primary review population.

The historical EIN-formatting issue was corrected, and Notebook 13 confirmed that historical financial information now populates correctly without export discrepancies.

The investigation queue preserves separate analytical rankings, financial and governance indicators, historical trends, peer comparisons, and explanations supporting investigative review.

The results also reinforce that supervised and anomaly-based approaches identify substantially different organizations.

**Decision:** Retain the corrected Notebook 12 as the completed investigation queue and explainability phase. Notebook 13 has validated the data pipeline and identified organization-size effects as the next major methodological concern.

Proceed to Notebook 14 to evaluate organization-size bias and model robustness before considering the investigation queue for operational deployment.

The project's objective remains identifying nonprofit organizations exhibiting potentially suspicious financial and governance patterns that may warrant further investigation—not predicting self-reported diversion disclosures alone.